# Lecture 1: Strings and the Genome

**Course:** Intro to Programming for Computational Biology  
**Prerequisites:** Variables, loops, if/else statements

In this notebook, we'll use Python to explore DNA — the molecule that encodes life. By the end, you'll be able to:
- Manipulate DNA sequences as Python strings
- Write a function that translates DNA into protein
- Use Biopython to fetch a real human gene from NCBI
- Simulate mutations and predict their effects

Our running example will be **HBB**, the gene encoding β-globin — a component of hemoglobin. A single mutation in this gene causes **sickle cell disease**.

---
## Section 1: DNA as a Python String

DNA is made of four nucleotide bases: **A** (adenine), **T** (thymine), **G** (guanine), and **C** (cytosine). A DNA sequence is just a string of these letters — which means we can use all our Python string tools on it!

In [ ]:
# A short example DNA sequence
dna = "ATGGTGCATCTGACTCCTGAGGAGAAGTCT"
print("Sequence:", dna)
print("Length:", len(dna))

### 1.1 Indexing and slicing

Like any Python string, we can index into a DNA sequence. Remember: Python uses **0-based indexing**.

In [ ]:
# What is the first nucleotide?
print(dna[0])

# What are the first 3 nucleotides (the first codon)?
print(dna[0:3])

# What is the last nucleotide?
print(dna[-1])

**Exercise 1.1** — Fill in the blanks to extract the **4th codon** (nucleotides 10–12, 1-based) from `dna`.

In [ ]:
# Hint: the 4th codon starts at index 9 (0-based)
fourth_codon = dna[___:___]
print("4th codon:", fourth_codon)  # Expected: CTG

### 1.2 Counting nucleotides

Python strings have a built-in `.count()` method.

In [ ]:
# Count each nucleotide
print("A:", dna.count("A"))
print("T:", dna.count("T"))
print("G:", dna.count("G"))
print("C:", dna.count("C"))

**Exercise 1.2** — Complete the function below to compute the **GC content** of a DNA sequence (the fraction of bases that are G or C). GC content is important because G-C base pairs are more stable than A-T pairs.

In [ ]:
def gc_content(sequence):
    """Return the GC content of a DNA sequence as a value between 0 and 1."""
    gc = sequence.count(___) + sequence.count(___)
    return gc / ___

print(gc_content(dna))  # Expected: ~0.567

### 1.3 Iterating over a sequence with a step

Slicing with a step (`sequence[start:stop:step]`) is useful for extracting every Nth element.

In [ ]:
# Print every 3rd nucleotide starting at position 0
print(dna[::3])

**Exercise 1.3** — Use a loop to split `dna` into a **list of codons** (groups of 3 nucleotides).

In [ ]:
codons = []
for i in range(___, ___, ___):
    codon = dna[___:___]
    codons.append(codon)

print(codons)
# Expected: ['ATG', 'GTG', 'CAT', 'CTG', 'ACT', 'CCT', 'GAG', 'GAG', 'AAG', 'TCT']

---
## Section 2: From DNA to Protein

Cells read DNA in groups of 3 bases called **codons**. Each codon specifies one amino acid (or a stop signal). This is the **genetic code**.

The process:
1. **Transcription**: DNA → mRNA (T becomes U)
2. **Translation**: mRNA codons → amino acids → protein

### 2.1 Transcription: DNA → mRNA

In [ ]:
# The .replace() method substitutes one substring for another
mrna = dna.replace("T", "U")
print("mRNA:", mrna)

### 2.2 The codon table

We'll represent the genetic code as a Python dictionary mapping codons to amino acids (single-letter codes). Stop codons are represented as `"*"`.

In [ ]:
codon_table = {
    # Phenylalanine (F)
    "UUU": "F", "UUC": "F",
    # Leucine (L)
    "UUA": "L", "UUG": "L", "CUU": "L", "CUC": "L", "CUA": "L", "CUG": "L",
    # Isoleucine (I)
    "AUU": "I", "AUC": "I", "AUA": "I",
    # Methionine / Start (M)
    "AUG": "M",
    # Valine (V)
    "GUU": "V", "GUC": "V", "GUA": "V", "GUG": "V",
    # Serine (S)
    "UCU": "S", "UCC": "S", "UCA": "S", "UCG": "S", "AGU": "S", "AGC": "S",
    # Proline (P)
    "CCU": "P", "CCC": "P", "CCA": "P", "CCG": "P",
    # Threonine (T)
    "ACU": "T", "ACC": "T", "ACA": "T", "ACG": "T",
    # Alanine (A)
    "GCU": "A", "GCC": "A", "GCA": "A", "GCG": "A",
    # Tyrosine (Y)
    "UAU": "Y", "UAC": "Y",
    # Stop
    "UAA": "*", "UAG": "*", "UGA": "*",
    # Histidine (H)
    "CAU": "H", "CAC": "H",
    # Glutamine (Q)
    "CAA": "Q", "CAG": "Q",
    # Asparagine (N)
    "AAU": "N", "AAC": "N",
    # Lysine (K)
    "AAA": "K", "AAG": "K",
    # Aspartate (D)
    "GAU": "D", "GAC": "D",
    # Glutamate (E)
    "GAA": "E", "GAG": "E",
    # Cysteine (C)
    "UGU": "C", "UGC": "C",
    # Tryptophan (W)
    "UGG": "W",
    # Arginine (R)
    "CGU": "R", "CGC": "R", "CGA": "R", "CGG": "R", "AGA": "R", "AGG": "R",
    # Serine (S) — already added above
    # Glycine (G)
    "GGU": "G", "GGC": "G", "GGA": "G", "GGG": "G",
    # Glutamate (E) — already added above
    # Aspartate (D) — already added above
}

print("Codons in table:", len(codon_table))

### 2.3 Translation

**Exercise 2.1** — Complete the `translate` function below. It should:
1. Convert the DNA sequence to mRNA (replace T with U)
2. Read the mRNA in codons of 3
3. Look up each codon in `codon_table`
4. Stop when it hits a stop codon (`"*"`)
5. Return the amino acid sequence as a string

In [ ]:
def translate(dna_sequence):
    """Translate a DNA sequence into a protein sequence."""
    mrna = dna_sequence.replace(___, ___)
    protein = ""
    for i in range(___, ___, ___):
        codon = mrna[___:___]
        amino_acid = codon_table.get(codon, "?")
        if amino_acid == ___:
            break
        protein += ___
    return protein

print(translate(dna))  # Expected: MVHLTPEE K S

---
## Section 3: Introducing Biopython

Biopython is a library that provides ready-made tools for biological sequence analysis. Let's see how it compares to what we just built — and use it to fetch the real HBB gene from NCBI's database.

In [ ]:
# Install biopython if needed (run once)
# !pip install biopython

In [ ]:
from Bio.Seq import Seq
from Bio import Entrez, SeqIO

# Always tell NCBI who you are
Entrez.email = "your.email@example.com"  # <-- replace with your email

### 3.1 Bio.Seq — the Biopython sequence object

In [ ]:
seq = Seq(dna)

print("Complement:        ", seq.complement())
print("Reverse complement:", seq.reverse_complement())
print("Transcription:     ", seq.transcribe())
print("Translation:       ", seq.translate())

Notice that `seq.translate()` produces the same amino acid sequence as our manual `translate()` function, but with the stop codon included as `*` at the end.

**Exercise 3.1** — Use `seq.translate()` to translate `dna`, then use string slicing to remove the trailing stop codon and compare to our manual result.

In [ ]:
bio_protein = str(seq.translate())
# Remove the trailing stop codon (*)
bio_protein_clean = bio_protein[___]

manual_result = translate(dna)
print("Biopython:", bio_protein_clean)
print("Manual:   ", manual_result)
print("Match:", bio_protein_clean == manual_result)

### 3.2 Fetching the real HBB gene from NCBI

The HBB gene (NCBI accene NM_000518) encodes the β-globin chain of hemoglobin. Let's download the real sequence.

In [ ]:
# Fetch the HBB mRNA record from NCBI
handle = Entrez.efetch(db="nucleotide", id="NM_000518", rettype="gb", retmode="text")
hbb_record = SeqIO.read(handle, "genbank")
handle.close()

print("Gene:", hbb_record.name)
print("Description:", hbb_record.description)
print("Sequence length:", len(hbb_record.seq), "bp")
print("First 60 bp:", hbb_record.seq[:60])

In [ ]:
# Extract the CDS (coding sequence) — the part that gets translated
for feature in hbb_record.features:
    if feature.type == "CDS":
        hbb_cds = feature.extract(hbb_record.seq)
        break

print("CDS length:", len(hbb_cds), "bp")
print("CDS:", hbb_cds)

In [ ]:
# Translate the CDS
hbb_protein = hbb_cds.translate(to_stop=True)
print("HBB protein (β-globin):")
print(hbb_protein)
print("Length:", len(hbb_protein), "amino acids")

**Exercise 3.2** — Use `.count()` on `hbb_protein` to find how many **glutamate (E)** residues are in β-globin. Position 6 (1-based) is the one that mutates in sickle cell disease.

In [ ]:
num_E = str(hbb_protein).count(___)
print("Glutamate (E) count:", num_E)
print("Amino acid at position 6:", str(hbb_protein)[___])  # 0-based index

---
## Section 4: Mutations

A **mutation** is a change in a DNA sequence. Three important types:

| Type | Effect on protein | Example |
|---|---|---|
| **Synonymous** | No change (same amino acid) | GAA → GAG (both = E) |
| **Missense** | Different amino acid | GAG → GTG (E → V) |
| **Nonsense** | Premature stop codon | GAG → TAG (E → stop) |

The sickle cell mutation is a missense mutation at codon 6 of HBB: **GAG → GTG** (glutamate → valine).

### 4.1 Introducing a point mutation

Strings in Python are **immutable** — we can't change a character in place. Instead, we use slicing to build a new string.

In [ ]:
def point_mutation(sequence, position, new_base):
    """
    Introduce a single-nucleotide substitution.
    position: 0-based index in the sequence
    new_base: the replacement nucleotide (A, T, G, or C)
    """
    return sequence[:position] + new_base + sequence[position + 1:]

# Test on a short sequence
test = "ATGAAGCTT"
print("Original: ", test)
print("Mutated:  ", point_mutation(test, 4, "T"))  # A→T at position 4

### 4.2 The sickle cell mutation

In HBB, codon 6 (0-based: positions 15–17 in the CDS) is GAG. The sickle cell mutation changes the **second base of this codon** from A to T: GAG → GTG.

In [ ]:
hbb_cds_str = str(hbb_cds)

# Codon 6 (1-based) = index 5 (0-based)
# In the CDS string, that's nucleotides at positions 15, 16, 17
print("Codon 6 (normal):  ", hbb_cds_str[15:18])  # Should be GAG

# The mutation changes position 16 (A → T)
hbb_sickle = point_mutation(hbb_cds_str, 16, "T")
print("Codon 6 (sickle):  ", hbb_sickle[15:18])  # Should be GTG

In [ ]:
# Translate both and compare
normal_protein  = Seq(hbb_cds_str).translate(to_stop=True)
sickle_protein  = Seq(hbb_sickle).translate(to_stop=True)

print("Normal position 6:  ", normal_protein[5])
print("Sickle position 6:  ", sickle_protein[5])
print("Proteins identical? ", normal_protein == sickle_protein)

**Exercise 4.1** — Complete the function below to classify a mutation as synonymous, missense, or nonsense. Then test it on the sickle cell mutation.

In [ ]:
def classify_mutation(original_cds, mutated_cds):
    """
    Classify a point mutation by comparing the translated proteins.
    Returns 'synonymous', 'missense', or 'nonsense'.
    """
    original_protein = str(Seq(original_cds).translate())
    mutated_protein  = str(Seq(mutated_cds).translate())

    # Check for nonsense: a stop codon (*) appears earlier in the mutated protein
    if ___ in mutated_protein and mutated_protein.index(___) < original_protein.index(___):
        return "nonsense"
    # Check for missense: proteins differ
    elif original_protein != ___:
        return "missense"
    # Otherwise: synonymous
    else:
        return ___

result = classify_mutation(hbb_cds_str, hbb_sickle)
print("Sickle cell mutation is:", result)  # Expected: missense

**Exercise 4.2** — Use `point_mutation` and `classify_mutation` to:
1. Create a **synonymous** mutation somewhere in the HBB CDS (change a nucleotide but keep the same amino acid)
2. Create a **nonsense** mutation (change a codon to a stop codon)

Print the position you chose, the nucleotide change, and the classification.

In [ ]:
# Your code here


**Exercise 4.3 (Open-ended)** — The table below lists three real HBB variants. For each one:
- Introduce the mutation using `point_mutation`
- Classify it
- Describe in one sentence what disease or effect it causes (look it up or ask your instructor)

| Variant | CDS position (0-based) | Change | Expected classification |
|---|---|---|---|
| HbC | 16 | A → G | missense |
| Hb Constant Spring (approx) | Try position 52 | G → A | nonsense |
| HbE | 76 | G → A | missense |

In [ ]:
# Your code here


---
## Section 5: Advanced Topics (Self-Study)

These exercises are for students who want to go further. They are not required.

### 5.1 Variant mapping

In clinical genetics, variants are often reported as **HGVS notation**, e.g., `c.20A>T` means position 20 in the CDS, A changed to T.

**Challenge:** Write a function `parse_hgvs(hgvs_string)` that parses a string like `"c.17A>T"` and returns the position (0-based), the reference base, and the alternate base. Then apply it to introduce the mutation programmatically.

In [ ]:
# Your code here


### 5.2 Motif finding

Transcription factors bind to specific short DNA sequences called **motifs**. For example, the TATA box motif is `TATAAAA` and is found in many gene promoters.

**Challenge:** Write a function `find_motif(sequence, motif)` that returns all **start positions** (0-based) where `motif` appears in `sequence`. Use it to search for the motif `"GCCNCC"` in the HBB promoter region (the 500 bp upstream of the CDS start).

*Hint: `N` matches any base — you may want to use the `re` module for this.*

In [ ]:
# Your code here
